## Data Pipelines - Prefect - Example and Workshop
### Big Data Tools 
#### M.Sc. in Applied Analytics (coterminal course)
Fac. de Ingeniería -  Universidad de la Sabana<br>
Prof.: Hugo Franco, Ph.D.

# Integrantes
## Juan David Henao
## Santiago Bazzani Rincón
## Pablo Boada Rozo

In [6]:
import os
os.environ['OMP_NUM_THREADS'] = '1'  # prevent problems with KMeans and Prefect

from functools import wraps
import time
from pathlib import Path

from prefect import flow, task, get_run_logger
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from kaggle import KaggleApi
import psycopg2
from psycopg2.extras import execute_values


# Measure the duration of each Prefect task.
def timing_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        logger = get_run_logger()
        started_at = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            elapsed = time.perf_counter() - started_at
            logger.info(f"Task '{func.__name__}' completed in {elapsed:.2f} seconds")
    return wrapper


# --- Configuration ---
DOWNLOAD_DIRECTORY = Path('data/')
TARGET_DATASET = 'madhur321/world-bank-development-indicators-panel'
PANEL_FILENAME = 'wdi_country_year_panel.csv'
TARGET_YEAR = 2022


@task(name='Download World Bank dataset', retries=3, retry_delay_seconds=10)
@timing_decorator
def download_dataset(download_path: Path, dataset_id: str) -> None:
    logger = get_run_logger()
    try:
        kaggle_api = KaggleApi()
        kaggle_api.authenticate()
        kaggle_api.dataset_download_files(
            dataset_id, path=str(download_path), unzip=True
        )
        logger.info(f"Downloaded dataset '{dataset_id}' to '{download_path}'.")
    except Exception as exc:
        logger.error(f"Kaggle download failed: {exc}")
        raise
    finally:
        logger.info('download_dataset task finished.')


@flow(name='extract-data')
def extract_data(
    download_path: Path = DOWNLOAD_DIRECTORY,
    dataset_id: str = TARGET_DATASET,
) -> Path:
    logger = get_run_logger()
    try:
        download_path = Path(download_path)
        download_path.mkdir(parents=True, exist_ok=True)
        logger.info(f"Starting dataset extraction into '{download_path}'.")
        download_dataset(download_path, dataset_id)
        return download_path
    except Exception as exc:
        logger.error(f"Data extraction flow failed: {exc}")
        raise
    finally:
        logger.info('extract_data flow finished.')


In [2]:
# Interpretable indicators used for clustering.
FEATURE_COLUMNS = [
    'gdp_per_capita_usd',
    'life_expectancy',
    'energy_use_per_capita',
    'internet_users_pct',
    'unemployment_pct',
    'fertility_rate',
]

# Defaults match the PostgreSQL container created in Session 2.
# Environment variables can override these values.
POSTGRES_CONFIG = {
    'host': os.getenv('PG_HOST', 'localhost'),
    'port': int(os.getenv('PG_PORT', '5433')),
    'dbname': os.getenv('PG_DB', 'bigdatatools1'),
    'user': os.getenv('PG_USER', 'psqluser'),
    'password': os.getenv('PG_PASS', 'psqlpass'),
}
POSTGRES_TABLE = 'wdi_country_clusters_2022'


@task(name='Read World Bank panel', retries=2, retry_delay_seconds=5)
@timing_decorator
def read_panel_data(path: Path) -> pd.DataFrame:
    logger = get_run_logger()
    try:
        if not path.is_file():
            raise FileNotFoundError(f"Dataset file not found: {path}")
        frame = pd.read_csv(path)
        logger.info(
            f"Raw panel loaded: {frame.shape[0]} rows x {frame.shape[1]} columns."
        )
        return frame
    except Exception as exc:
        logger.error(f"Could not read the World Bank panel: {exc}")
        raise
    finally:
        logger.info('read_panel_data task finished.')


@task(name='Transform data', retries=2, retry_delay_seconds=5)
@timing_decorator
def transform_data(df_wdi: pd.DataFrame, target_year: int = TARGET_YEAR) -> pd.DataFrame:
    logger = get_run_logger()
    try:
        required_columns = ['year', *FEATURE_COLUMNS]
        missing_columns = [column for column in required_columns if column not in df_wdi]
        if missing_columns:
            raise ValueError(f"Missing required columns: {missing_columns}")

        df_countries = df_wdi.loc[df_wdi['year'] == target_year].copy()
        logger.info(f"Year {target_year} snapshot: {len(df_countries)} countries.")
        df_clustering = (
            df_countries[FEATURE_COLUMNS]
            .dropna()
            .reset_index(drop=True)
        )
        n_dropped = len(df_countries) - len(df_clustering)
        logger.info(
            f"Clustering frame: {len(df_clustering)} complete countries "
            f"({n_dropped} dropped due to missing indicators)."
        )
        if len(df_clustering) < 3:
            raise ValueError('At least three complete countries are required for clustering.')
        return df_clustering
    except Exception as exc:
        logger.error(f"Data transformation failed: {exc}")
        raise
    finally:
        logger.info('transform_data task finished.')


@flow(name='prepare-clustering-data')
def prepare_clustering_data_flow(
    panel_path: Path,
    target_year: int = TARGET_YEAR,
) -> pd.DataFrame:
    logger = get_run_logger()
    try:
        logger.info('Starting the data preparation flow.')
        raw_panel = read_panel_data(Path(panel_path))
        clustering_frame = transform_data(raw_panel, target_year)
        logger.info('Data preparation flow completed successfully.')
        return clustering_frame
    except Exception as exc:
        logger.error(f"Data preparation flow failed: {exc}")
        raise
    finally:
        logger.info('prepare_clustering_data_flow finished.')


@task(name='Load clustering results to PostgreSQL', retries=2, retry_delay_seconds=5)
@timing_decorator
def load_data(df_clustered: pd.DataFrame) -> int:
    logger = get_run_logger()
    conn = None
    try:
        required_columns = [*FEATURE_COLUMNS, 'cluster']
        missing_columns = [column for column in required_columns if column not in df_clustered]
        if missing_columns:
            raise ValueError(f"Cannot load data; missing columns: {missing_columns}")

        conn = psycopg2.connect(**POSTGRES_CONFIG)
        column_definitions = ', '.join(
            f'"{column}" INTEGER' if column == 'cluster'
            else f'"{column}" DOUBLE PRECISION'
            for column in required_columns
        )
        insert_columns = ', '.join(f'"{column}"' for column in required_columns)
        records = [
            tuple(
                None if pd.isna(value)
                else value.item() if isinstance(value, np.generic)
                else value
                for value in row
            )
            for row in df_clustered[required_columns].itertuples(index=False, name=None)
        ]

        with conn:
            with conn.cursor() as cursor:
                cursor.execute(
                    f"CREATE TABLE IF NOT EXISTS public.{POSTGRES_TABLE} "
                    f"(id BIGSERIAL PRIMARY KEY, {column_definitions})"
                )
                cursor.execute(f'TRUNCATE TABLE public.{POSTGRES_TABLE}')
                execute_values(
                    cursor,
                    f"INSERT INTO public.{POSTGRES_TABLE} ({insert_columns}) VALUES %s",
                    records,
                )
        logger.info(
            f"Loaded {len(records)} rows into public.{POSTGRES_TABLE}."
        )
        return len(records)
    except Exception as exc:
        if conn is not None:
            conn.rollback()
        logger.error(f"Could not load clustering results into PostgreSQL: {exc}")
        raise
    finally:
        if conn is not None:
            conn.close()
        logger.info('load_data task finished.')


@flow(name='world-bank-clustering-end-to-end')
def end_to_end_pipeline_flow(
    target_year: int = TARGET_YEAR,
    download_path: Path = DOWNLOAD_DIRECTORY,
) -> dict:
    logger = get_run_logger()
    try:
        logger.info('Starting the end-to-end World Bank clustering pipeline.')
        dataset_path = extract_data(Path(download_path), TARGET_DATASET)
        df_clustering = prepare_clustering_data_flow(
            Path(dataset_path) / PANEL_FILENAME, target_year
        )
        dataset_size_bytes = report_download_size(Path(dataset_path))
        correlation_matrix, lowest_corr_pair = analyze_feature_correlations(df_clustering)
        (
            df_clustered,
            features,
            _X,
            optimal_k,
            _kmeans_model,
            silhouette_scores,
        ) = run_kmeans_analysis(df_clustering, lowest_corr_pair)
        loaded_rows = load_data(df_clustered)
        best_silhouette = silhouette_scores[optimal_k]
        summary = {
            'year': target_year,
            'countries_loaded': loaded_rows,
            'selected_features': list(features),
            'optimal_k': optimal_k,
            'best_silhouette': round(float(best_silhouette), 4),
            'dataset_size_bytes': dataset_size_bytes,
            'postgres_table': f'public.{POSTGRES_TABLE}',
        }
        logger.info(f"Pipeline summary: {summary}")
        return summary
    except Exception as exc:
        logger.error(f"End-to-end pipeline failed: {exc}")
        raise
    finally:
        logger.info('End-to-end pipeline finished.')


### Unsupervised machine learning

Determine a useful number of country groups using World Bank development indicators.

First identify the least correlated pair of indicators and use them to represent each country.

In [ ]:
@task(name="Analyze feature correlations")
@timing_decorator
def analyze_feature_correlations(df_clustering: pd.DataFrame):
    logger = get_run_logger()
    try:
        # Calculate the correlation matrix.
        correlation_matrix = df_clustering.corr()

        # Plot a heatmap of the correlation matrix for visualization.
        plt.figure(figsize=(10, 8))
        sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
        plt.title('Correlation Matrix of World Bank Indicators')
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.savefig('correlation_heatmap.png')
        plt.close()   # release the figure

        # To find the lowest correlation, we need to ignore the diagonal (self-correlation) and look 
        # for the smallest absolute value in the matrix (variable selection for K-Means analysis)
        # 1. Convert to absolute values (strength matters, not direction)
        abs_corr_matrix = correlation_matrix.abs()

        # 2. "Unstack" the matrix into a long list of pairs
        # This turns a [Variable x Variable] table into a [Variable Pair | Value] list
        all_pairs = abs_corr_matrix.unstack()

        # 3. Filter out the diagonal (where a variable correlates with itself)
        # Since self-correlation is always 1.0, we just look for anything less than 1
        unique_pairs = all_pairs[all_pairs < 1]

        # 4. Find the smallest value and its labels
        lowest_corr = unique_pairs.min()
        lowest_corr_pair = unique_pairs.idxmin()

        logger.info(f"The lowest absolute correlation is {lowest_corr:.4f} between {lowest_corr_pair}")
        return correlation_matrix, lowest_corr_pair
    except Exception as exc:
        logger.error(f"Correlation analysis failed: {exc}")
        raise
    finally:
        logger.info("analyze_feature_correlations task finished.")



Now use `KMeans` to identify groups of countries from the selected World Bank indicators. Compare the elbow and silhouette results before choosing the number of clusters.

In [ ]:
@task(name="Select and run KMeans")
@timing_decorator
def run_kmeans_analysis(
    df_clustering: pd.DataFrame,
    lowest_corr_pair,
):
    logger = get_run_logger()
    try:
        # --- 1. FEATURE SELECTION ---
        features = list(lowest_corr_pair)
        X = df_clustering[features]


        # --- 2. FUNCTIONS: ELBOW AND SILHOUETTE ---

        def find_elbow(inertia: list, min_k: int = 2) -> int:
            """
            Geometric elbow detection. Skips K < min_k because a single cluster has
            no analytical value and the K=1 -> K=2 inertia drop always dominates the
            geometry, producing a spurious 'K = 1' answer on flat curves.
            """
            k_vals = np.arange(1, len(inertia) + 1, dtype=float)
            coords = np.column_stack((k_vals, inertia))
            p1, p2 = coords[0], coords[-1]
            chord = p2 - p1
            chord /= np.linalg.norm(chord)
            ortho = np.array([-chord[1], chord[0]])
            distances = np.abs((coords - p1) @ ortho)
            distances[: min_k - 1] = -np.inf       # exclude K = 1 (and K < min_k)
            return int(k_vals[np.argmax(distances)])


        def silhouette_sweep(X, k_values) -> dict:
            """
            Mean silhouette score for each K in k_values (K >= 2 only).
            Range: -1 .. 1. Higher = better separated & more compact clusters.
              > 0.50  strong structure
              0.25–0.50  moderate structure
              < 0.25  weak / no structure (a red flag for this feature pair)
            """
            scores = {}
            for k in k_values:
                if k < 2:
                    continue                    # silhouette undefined for K = 1
                labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X)
                scores[k] = silhouette_score(X, labels)
            return scores


        def find_optimal_k_by_silhouette(scores: dict) -> int:
            """Return the K that maximises the mean silhouette score."""
            return max(scores, key=scores.get)


        # --- 3. RUN BOTH SWEEPS ---
        k_values = list(range(1, 11))

        inertia = []
        for k in k_values:
            km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
            inertia.append(km.inertia_)

        silhouette_scores = silhouette_sweep(X, k_values)


        # --- 4. COMBINED PLOT ---
        fig, ax1 = plt.subplots(figsize=(9, 6))

        # Left axis: inertia (elbow)
        ax1.plot(k_values, inertia, marker='o', color='steelblue', label='Inertia (elbow)')
        ax1.set_xlabel('Number of Clusters (K)')
        ax1.set_ylabel('Inertia', color='steelblue')
        ax1.tick_params(axis='y', labelcolor='steelblue')
        ax1.set_xticks(k_values)
        ax1.grid(True, alpha=0.3)

        # Right axis: silhouette
        ax2 = ax1.twinx()
        sil_ks = sorted(silhouette_scores)
        sil_vals = [silhouette_scores[k] for k in sil_ks]
        ax2.plot(sil_ks, sil_vals, marker='s', linestyle='--',
                 color='darkorange', label='Silhouette')
        ax2.set_ylabel('Silhouette score', color='darkorange')
        ax2.tick_params(axis='y', labelcolor='darkorange')

        # Combined legend
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')

        fig.suptitle(f'Elbow + Silhouette ({features[0]} vs {features[1]})')
        plt.tight_layout()
        plt.savefig('elbow_silhouette_plot.png')
        plt.close(fig)


        # --- 5. COMBINE SIGNALS INTO A SINGLE DECISION ---
        k_elbow = find_elbow(inertia, min_k=2)
        k_sil   = find_optimal_k_by_silhouette(silhouette_scores)
        best_sil = silhouette_scores[k_sil]

        logger.info(f"Elbow suggests:      K = {k_elbow}")
        logger.info(f"Silhouette suggests: K = {k_sil}   (best score = {best_sil:.3f})")
        logger.info("Full silhouette sweep: %s", {k: round(v, 3) for k, v in silhouette_scores.items()})

        if k_elbow == k_sil:
            optimal_k = k_elbow
            logger.info(f"→ Both methods agree: K = {optimal_k}.")
        else:
            optimal_k = k_sil
            logger.info(
                f"→ Disagreement (elbow={k_elbow}, silhouette={k_sil}). "
                f"Defaulting to silhouette (quantitative separation metric). "
                f"Inspect the plot and override manually if domain reasoning favours "
                f"another K."
            )

        if best_sil < 0.25:
            logger.info(
                "⚠  Best silhouette is below 0.25 → this feature pair does not cluster "
                "well. Expect diffuse clouds, not clean blobs. This is itself a valid "
                "analytical finding (see workshop Analysis question)."
            )

        # Manual override example (uncomment to force a value):
        # optimal_k = 3


        # --- 6. K-MEANS EXECUTION ---
        kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
        cluster_labels = kmeans_model.fit_predict(X)
        df_clustering = df_clustering.assign(cluster=cluster_labels)


        # --- 7. VISUALIZATION ---
        fig, ax = plt.subplots(figsize=(10, 8))
        scatter = ax.scatter(
            X.iloc[:, 0],
            X.iloc[:, 1],
            c=cluster_labels,
            cmap='viridis',
            alpha=0.85,
        )
        centroids = kmeans_model.cluster_centers_
        centroid_scatter = ax.scatter(
            centroids[:, 0],
            centroids[:, 1],
            marker='X',
            s=240,
            c='red',
            edgecolors='black',
            linewidths=1.2,
            label='Centroids',
            zorder=3,
        )
        cluster_handles, _ = scatter.legend_elements()
        cluster_legend_labels = [f'Cluster {index}' for index in range(optimal_k)]
        ax.legend(
            cluster_handles + [centroid_scatter],
            cluster_legend_labels + ['Centroids'],
            title='Groups',
        )
        ax.set_title(
            f'K-means Clustering (K={optimal_k}, '
            f'silhouette={silhouette_scores[optimal_k]:.3f})'
        )
        ax.set_xlabel(features[0])
        ax.set_ylabel(features[1])
        ax.grid(True)
        plt.tight_layout()
        plt.savefig('kmeans_clusters.png', dpi=150)
        plt.close(fig)
        return (
            df_clustering,
            features,
            X,
            optimal_k,
            kmeans_model,
            silhouette_scores,
        )
    except Exception as exc:
        logger.error(f"KMeans analysis failed: {exc}")
        raise
    finally:
        logger.info("run_kmeans_analysis task finished.")



__Challenges (Workshop 3, part I):__
1. Organize the code of the example and complete the data pipeline: 
    - Use `try-except-finally` blocks as required 
    - Every individual process must be wrapped as a `task` using the corresponding decorator and its parameters when necessary. 
    - Create a `transform_data` task focused on data cleansing and preparation (variable selection/deletion)
    - Create a `load_data` task to create a table in a Dockerized PostgreSQL database and populate with with the clustering-oriented Dataframe
    - Invoke the tasks in the proper order in the `flow`
    - Use the `timing_decorator` to report the duration of each task
    - **Include the analytic part of this notebook (model selection, visualization) to implement an End-to-End pipeline (flow)**
1. Extend the filesystem report
    - Using the function `get_directory_size` (below), create a decorator to get and report the size of the downloaded dataset (size in bytes of the download folder)
    - Report the size in the previous question in a human readable unit
    - Use the attribute kmeans.cluster_centers_ and scatter plot to add the centroids of the best KMeans model (best K parameter) to the scatter plot
        - `centroids = kmeans.cluster_centers_`
1. Analysis
    - Based on the original data, justify the result of the selection of the  parameter K for the K-means method

In [ ]:
def get_directory_size(path='.'):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for filename in filenames:
            file_path = os.path.join(dirpath, filename)
            if not os.path.islink(file_path):
                total_size += os.path.getsize(file_path)
    return total_size


def human_readable_size(size_bytes: int) -> str:
    size = float(size_bytes)
    for unit in ('B', 'KB', 'MB', 'GB', 'TB'):
        if size < 1024 or unit == 'TB':
            return f'{size:.2f} {unit}'
        size /= 1024


def report_directory_size(func):
    @wraps(func)
    def wrapper(path, *args, **kwargs):
        logger = get_run_logger()
        try:
            size_bytes = func(path, *args, **kwargs)
            logger.info(
                f"Downloaded dataset folder size: {size_bytes:,} bytes "
                f"({human_readable_size(size_bytes)})."
            )
            return size_bytes
        except Exception as exc:
            logger.error(f"Could not report dataset folder size: {exc}")
            raise
        finally:
            logger.info('Dataset size report finished.')
    return wrapper


@task(name='Report downloaded dataset size')
@timing_decorator
@report_directory_size
def report_download_size(path: Path) -> int:
    return get_directory_size(path)


pipeline_results = end_to_end_pipeline_flow()

pipeline_results
